# Entity ***Ticks***
+ Layer **bronze**
+ Priority **2**
---
### Imports

In [0]:
%run ../common/bronze_constants

In [0]:
%run ../common/medallion_functions

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Subgraph API

In [0]:
SUBGRAPH_URL = get_subgraph_api_url_from_secrets()

### Variables

In [0]:
ticks_list = []
query_variables= {
    "batch_pool_ids": [],
    "skip": 0
}

In [0]:
ticks_query = """
query TicksQueryBatch($batch_pool_ids: [String!]!, $skip: Int!) {
  ticks(
    where: {
      pool_in: $batch_pool_ids,
      liquidityNet_not: "0",
      liquidityGross_not: "0"
    },
    first: 1000,
    skip: $skip, 
  ){
    id
    poolAddress
    tickIdx
    pool {
      id
    }
    volumeToken0
    volumeToken1
    volumeUSD
    price0
    price1
    liquidityGross
    liquidityNet
    feesUSD
    collectedFeesToken0
    collectedFeesToken1
    collectedFeesUSD
    createdAtTimestamp
    createdAtBlockNumber
  }
}
"""

### Filtering relevant pools
+ Ticks could overload the graph queries threshold since within a single pools could exists a large amount of ticks.

In [0]:
filtered_bz_pools_df = retrieve_relevant_bronze_pools()

### Execution

In [0]:
pool_ids_list = [row.id for row in filtered_bz_pools_df.select("id").collect()]

In [0]:
for i in range(0, len(pool_ids_list), BATCH_SIZE):
    batch_pools = pool_ids_list[i:i+BATCH_SIZE]
    query_variables["batch_pool_ids"] = batch_pools
    query_variables["skip"] = 0
    print(f"*INFO*: Loading ticks for {len(batch_pools)} pools.")
    while True:
        response = requests.post(SUBGRAPH_URL, json={"query": ticks_query, "variables": query_variables})
        ticks_data = response.json()["data"][entity_name]

        if "errors" in response.json():
            raise Exception(response.json()["errors"])
        
        ticks_list.extend(ticks_data)

        time.sleep(0.5)

        rows_loaded = len(ticks_data)
        print(f"*INFO*: Loaded rows: {rows_loaded}.")
        
        if rows_loaded < PAGE_SIZE:
            break
           
        query_variables["skip"] += PAGE_SIZE
        

In [0]:
ticks_df = spark.createDataFrame(ticks_list)
ticks_df = ticks_df.withColumn("_load_timestamp_bronze", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(ticks_df,
                        entity_name,
                        load_pattern,
                        layer
                        #,
                        )

In [0]:
dbutils.notebook.exit(load_result)